# Exploratory Data Analysis

本 Notebook 用于机器学习课程设计项目的数据探索。当前阶段只进行数据读取、结构检查、标签分布、缺失值、唯一值、重复值和时间字段分析，不进行任何模型训练。

## 1. 环境准备

导入基础数据分析库，并创建输出目录。

In [ ]:
# 导入数据处理、数值计算和绘图库，不使用 seaborn
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 设置图表基础样式，保证中文环境下尽量正常显示
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

# Notebook 设计为从项目根目录运行；如果当前目录没有 data，则尝试兼容从 notebooks 目录运行
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# 定义数据目录和输出目录，输出目录不存在时自动创建
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"项目根目录: {PROJECT_ROOT}")
print(f"数据目录: {DATA_DIR}")
print(f"输出目录: {OUTPUT_DIR}")

## 2. 数据读取

读取训练集、测试集和提交样例文件。文件名优先匹配带 `(1)` 的版本，同时兼容不带 `(1)` 的版本。

In [ ]:
# 定义文件查找函数，用于兼容不同数据文件命名
def find_data_file(candidates):
    """按照候选文件名顺序查找数据文件。"""
    for filename in candidates:
        file_path = DATA_DIR / filename
        if file_path.exists():
            return file_path
    raise FileNotFoundError(f"未在 {DATA_DIR} 找到候选文件: {candidates}")


# 定义自动识别分隔符的读取函数，优先使用 Tab 分隔，列数异常时再尝试逗号分隔
def read_csv_auto_sep(file_path):
    """自动识别 Tab 或逗号分隔的 CSV 文件。"""
    df = pd.read_csv(file_path, sep="\t")
    used_sep = "Tab"
    
    # 如果按 Tab 读取后仍只有一列，说明可能是逗号分隔，再重新读取
    if df.shape[1] == 1:
        comma_df = pd.read_csv(file_path, sep=",")
        if comma_df.shape[1] > df.shape[1]:
            df = comma_df
            used_sep = "Comma"
    
    # 清理字段名前后空白，避免后续字段匹配失败
    df.columns = df.columns.astype(str).str.strip()
    print(f"{file_path.name} 使用分隔符: {used_sep}, 读取列数: {df.shape[1]}")
    return df


# 查找训练集、测试集和提交样例文件
train_path = find_data_file(["train_dataset(1).csv", "train_dataset.csv"])
test_path = find_data_file(["test_dataset(1).csv", "test_dataset.csv"])
submit_path = find_data_file(["submit_example(1).csv", "submit_example.csv"])

# 读取三个 CSV 文件，自动处理 Tab 或逗号分隔
train = read_csv_auto_sep(train_path)
test = read_csv_auto_sep(test_path)
submit = read_csv_auto_sep(submit_path)

# 检查关键字段是否已经被正确拆分为独立字段
expected_train_columns = {"session_id", "op_date", "risk_label"}
missing_train_columns = expected_train_columns - set(train.columns)
if missing_train_columns:
    raise ValueError(f"训练集关键字段未正确读取，请检查分隔符或表头: {sorted(missing_train_columns)}")

# 保存原始数据规模，避免后续新增 EDA 派生字段影响结论统计
raw_train_shape = train.shape
raw_test_shape = test.shape
raw_submit_shape = submit.shape

print(f"训练集文件: {train_path.name}")
print(f"测试集文件: {test_path.name}")
print(f"提交样例文件: {submit_path.name}")

## 3. 数据基本信息

查看数据规模、前几行样本和字段类型。

In [ ]:
# 输出训练集、测试集和提交样例的数据规模
print("train.shape:", train.shape)
print("test.shape:", test.shape)
print("submit.shape:", submit.shape)

In [ ]:
# 查看训练集前 5 行
train.head()

In [ ]:
# 查看测试集前 5 行
test.head()

In [ ]:
# 输出训练集字段类型和非空数量
train.info()

In [ ]:
# 输出测试集字段类型和非空数量
test.info()

## 4. 字段差异检查

比较训练集和测试集字段差异，并确认目标字段 `risk_label` 只存在于训练集。

In [ ]:
# 对比训练集和测试集字段集合
train_columns = set(train.columns)
test_columns = set(test.columns)

# 找出只存在于训练集或只存在于测试集的字段
only_in_train = sorted(train_columns - test_columns)
only_in_test = sorted(test_columns - train_columns)

print("只存在于训练集的字段:", only_in_train)
print("只存在于测试集的字段:", only_in_test)

# 确认 risk_label 是否只存在于训练集
risk_label_only_in_train = "risk_label" in train_columns and "risk_label" not in test_columns
print("risk_label 是否只存在于训练集:", risk_label_only_in_train)

# 如果训练集中没有 risk_label，后续标签分析无法继续，直接给出明确提示
if "risk_label" not in train_columns:
    raise KeyError("训练集中未找到 risk_label 字段，请检查数据文件。")

## 5. 标签分布分析

统计 `risk_label` 的取值数量、比例和正负样本比例，并保存标签分布柱状图。

In [ ]:
# 统计 risk_label 的数量和占比
label_counts = train["risk_label"].value_counts(dropna=False).sort_index()
label_ratios = train["risk_label"].value_counts(normalize=True, dropna=False).sort_index()
label_summary = pd.DataFrame({
    "count": label_counts,
    "ratio": label_ratios,
})

print("risk_label 标签分布:")
display(label_summary)

# 如果标签为常见的 0/1 二分类形式，则单独输出正负样本比例
label_values = set(train["risk_label"].dropna().unique())
if {0, 1}.issubset(label_values):
    positive_count = int((train["risk_label"] == 1).sum())
    negative_count = int((train["risk_label"] == 0).sum())
    positive_ratio = positive_count / len(train)
    negative_ratio = negative_count / len(train)
    print(f"正样本数量: {positive_count}, 占比: {positive_ratio:.4%}")
    print(f"负样本数量: {negative_count}, 占比: {negative_ratio:.4%}")
    if negative_count > 0:
        print(f"正负样本比例（1:0）: {positive_count / negative_count:.4f}:1")
else:
    print("risk_label 不是标准 0/1 标签，已输出各类别数量和比例。")

# 绘制并保存标签分布柱状图
fig, ax = plt.subplots(figsize=(6, 4))
label_counts.plot(kind="bar", ax=ax, color="#4C78A8")
ax.set_title("risk_label 标签分布")
ax.set_xlabel("risk_label")
ax.set_ylabel("样本数量")
plt.xticks(rotation=0)
plt.tight_layout()

label_distribution_path = OUTPUT_DIR / "label_distribution.png"
plt.savefig(label_distribution_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"标签分布图已保存至: {label_distribution_path}")

## 6. 缺失值统计

分别统计训练集和测试集每个字段的缺失值数量及比例。

In [ ]:
# 定义缺失值统计函数，返回每个字段的缺失数量和缺失比例
def missing_summary(df):
    missing_count = df.isna().sum()
    missing_ratio = missing_count / len(df)
    return pd.DataFrame({
        "missing_count": missing_count,
        "missing_ratio": missing_ratio,
    }).sort_values("missing_count", ascending=False)


# 统计训练集缺失值
train_missing = missing_summary(train)
print("训练集缺失值统计:")
display(train_missing)

# 统计测试集缺失值
test_missing = missing_summary(test)
print("测试集缺失值统计:")
display(test_missing)

## 7. 唯一值数量统计

统计训练集和测试集每个字段的唯一值数量。

In [ ]:
# 统计训练集每个字段的唯一值数量，包含缺失值作为一种取值
train_nunique = train.nunique(dropna=False).sort_values(ascending=False)
print("训练集唯一值数量:")
display(train_nunique.to_frame("nunique"))

# 统计测试集每个字段的唯一值数量，包含缺失值作为一种取值
test_nunique = test.nunique(dropna=False).sort_values(ascending=False)
print("测试集唯一值数量:")
display(test_nunique.to_frame("nunique"))

## 8. session_id 分析

检查训练集和测试集中的 `session_id` 是否唯一，并统计重复情况。

In [ ]:
# 检查 session_id 字段是否存在
if "session_id" not in train.columns or "session_id" not in test.columns:
    print("训练集或测试集中缺少 session_id 字段，请检查字段名称。")
else:
    # 判断训练集和测试集 session_id 是否唯一
    train_session_unique = train["session_id"].is_unique
    test_session_unique = test["session_id"].is_unique
    
    # 统计重复 session_id 数量
    train_session_duplicates = int(train["session_id"].duplicated().sum())
    test_session_duplicates = int(test["session_id"].duplicated().sum())
    
    print("训练集 session_id 是否唯一:", train_session_unique)
    print("测试集 session_id 是否唯一:", test_session_unique)
    print("训练集重复 session_id 行数:", train_session_duplicates)
    print("测试集重复 session_id 行数:", test_session_duplicates)
    
    # 如存在重复，展示重复最多的 session_id
    if train_session_duplicates > 0:
        print("训练集中重复最多的 session_id:")
        display(train["session_id"].value_counts().loc[lambda x: x > 1].head())
    if test_session_duplicates > 0:
        print("测试集中重复最多的 session_id:")
        display(test["session_id"].value_counts().loc[lambda x: x > 1].head())

## 9. op_date 时间字段分析

将 `op_date` 转换为 datetime，并提取 year、month、day、hour、weekday 等时间特征，统计小时分布并保存图表。

In [ ]:
# 检查 op_date 字段是否存在
if "op_date" not in train.columns or "op_date" not in test.columns:
    print("训练集或测试集中缺少 op_date 字段，请检查字段名称。")
else:
    # 将 op_date 转换为 datetime，无法解析的值会变为 NaT
    train["op_date"] = pd.to_datetime(train["op_date"], errors="coerce")
    test["op_date"] = pd.to_datetime(test["op_date"], errors="coerce")
    
    # 统计无法转换为日期时间的记录数量
    print("训练集 op_date 无法解析数量:", int(train["op_date"].isna().sum()))
    print("测试集 op_date 无法解析数量:", int(test["op_date"].isna().sum()))
    
    # 提取训练集和测试集的时间特征
    for df in [train, test]:
        df["op_year"] = df["op_date"].dt.year
        df["op_month"] = df["op_date"].dt.month
        df["op_day"] = df["op_date"].dt.day
        df["op_hour"] = df["op_date"].dt.hour
        df["op_weekday"] = df["op_date"].dt.weekday
    
    # 统计训练集和测试集不同小时的样本数量
    hours = list(range(24))
    train_hour_counts = train["op_hour"].value_counts().reindex(hours, fill_value=0).sort_index()
    test_hour_counts = test["op_hour"].value_counts().reindex(hours, fill_value=0).sort_index()
    hour_summary = pd.DataFrame({
        "train_count": train_hour_counts,
        "test_count": test_hour_counts,
    })
    
    print("不同小时样本数量统计:")
    display(hour_summary)
    
    # 绘制并保存小时分布图
    ax = hour_summary.plot(kind="bar", figsize=(10, 4), color=["#4C78A8", "#F58518"])
    ax.set_title("op_date 小时分布")
    ax.set_xlabel("小时")
    ax.set_ylabel("样本数量")
    plt.xticks(rotation=0)
    plt.tight_layout()
    
    hour_distribution_path = OUTPUT_DIR / "hour_distribution.png"
    plt.savefig(hour_distribution_path, dpi=150, bbox_inches="tight")
    plt.show()
    
    print(f"小时分布图已保存至: {hour_distribution_path}")

## 10. 简短结论

基于以上探索结果，总结数据规模、任务类型、标签不平衡情况和后续预处理重点。

In [ ]:
# 根据 risk_label 的唯一值数量判断任务类型
label_unique_count = train["risk_label"].nunique(dropna=True)
if label_unique_count == 2:
    task_type = "二分类任务"
else:
    task_type = f"监督学习任务，当前标签类别数为 {label_unique_count}"

# 判断标签是否存在明显不平衡，这里用少数类占比低于 40% 作为初步判断标准
valid_label_counts = train["risk_label"].value_counts(dropna=True)
if len(valid_label_counts) > 0:
    minority_ratio = valid_label_counts.min() / valid_label_counts.sum()
    imbalance_text = "存在一定不平衡" if minority_ratio < 0.4 else "整体较为均衡"
else:
    minority_ratio = np.nan
    imbalance_text = "无法判断，标签为空"

# 根据缺失值、重复 ID、时间字段和高基数字段整理后续预处理重点
preprocess_points = []
if train_missing["missing_count"].sum() > 0 or test_missing["missing_count"].sum() > 0:
    preprocess_points.append("处理缺失值")
if "session_id" in train.columns and train["session_id"].duplicated().sum() > 0:
    preprocess_points.append("排查训练集重复 session_id")
if "session_id" in test.columns and test["session_id"].duplicated().sum() > 0:
    preprocess_points.append("排查测试集重复 session_id")
if "op_date" in train.columns and "op_hour" in train.columns:
    preprocess_points.append("整理 op_date 时间特征")

# 初步识别唯一值数量接近样本数的高基数字段，后续需要谨慎处理
high_cardinality_columns = train_nunique[train_nunique > len(train) * 0.8].index.tolist()
if high_cardinality_columns:
    preprocess_points.append("评估高基数字段的编码或剔除策略")

if not preprocess_points:
    preprocess_points.append("继续结合业务含义进行特征清洗和特征工程")

# 输出简短结论，不包含任何模型训练结果
print("简短结论:")
print(f"1. 数据规模：训练集 {raw_train_shape[0]} 行、{raw_train_shape[1]} 列；测试集 {raw_test_shape[0]} 行、{raw_test_shape[1]} 列；提交样例 {raw_submit_shape[0]} 行、{raw_submit_shape[1]} 列。")
print(f"2. 任务类型：{task_type}，目标字段为 risk_label，且 risk_label 只应存在于训练集。")
if pd.notna(minority_ratio):
    print(f"3. 标签是否不平衡：{imbalance_text}，少数类占比约为 {minority_ratio:.2%}。")
else:
    print(f"3. 标签是否不平衡：{imbalance_text}。")
print(f"4. 后续预处理重点：{'；'.join(preprocess_points)}。")
print("5. 当前 Notebook 仅完成数据探索，未进行模型训练。")